<a href="https://colab.research.google.com/github/sangchun1/Blackbox-Detection/blob/stage1-sangchun/scripts/stage1/01_train_videomaev2_b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage 1 - Phase 1: V1 VideoMAEv2-B (video branch)

Trains VideoMAEv2-B using the team's fixed Stage 1 CSVs.

Current data policy:
- DLC-2021 supplies both ORIGINAL and RERECORDED supervision.
- The currently available CCD Crash/Normal videos are **not re-recorded**, so when
  they are included in `train.csv` they must use the Stage 1 label `ORIGINAL`.
- `Crash` / `Normal` are scene labels only (optional `scene_type`), never Stage 1 labels.
- Until physical CCD re-recordings exist, checkpoint selection and threshold tuning
  use the DLC-2021 portion of `val.csv` only.
- `test.csv` is kept untouched as the final internal holdout.

The notebook only orchestrates; model/data logic lives in `blackbox_detection.stage1`.

**Current data source:** `DATASET/DLC-2021/dlc_split.csv` and `DATASET/CCD/ccd_split.csv`. The notebooks do not expect a `stage1_splits/` directory. W&B authenticates from `MyDrive/Blackbox-Detection/wandb_key.txt`.

**Colab environment note:** this version does not install the full `pyproject.toml` dependency set into the live kernel. It preserves Colab's NumPy/SciPy/PyTorch stack, installs only Stage 1 extras, then installs the repository with `--no-deps` to avoid binary-package mismatch errors such as `No module named numpy.rec`.

**DLC path note:** actual videos are indexed from `DLC-2021/or/clips_video/**` and `DLC-2021/re/clips_video/**`. `or/clips/annotations/**` and `re/clips/annotations/**` are JSON annotations and are ignored.


## 1. Setup

In [1]:
from __future__ import annotations

import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

# ============================================================
# Colab + GitHub + Google Drive setup
# ============================================================
REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage1-sangchun"

IN_COLAB = False
try:
    from google.colab import drive

    IN_COLAB = True
    drive.mount("/content/drive", force_remount=False)
except ModuleNotFoundError:
    print("Not running in Google Colab; Drive mount skipped.")

if IN_COLAB:
    REPO_ROOT = Path("/content/Blackbox-Detection")

    if not (REPO_ROOT / ".git").is_dir():
        print(f"Cloning {REPO_URL} [{BRANCH}] -> {REPO_ROOT}")
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                BRANCH,
                "--single-branch",
                REPO_URL,
                str(REPO_ROOT),
            ],
            check=True,
        )
    else:
        current_branch = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "branch", "--show-current"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip()

        if current_branch != BRANCH:
            subprocess.run(
                ["git", "-C", str(REPO_ROOT), "checkout", BRANCH],
                check=True,
            )

        dirty = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "status", "--porcelain"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip()

        if dirty:
            print(
                "WARNING: /content/Blackbox-Detection has local changes. "
                "Automatic git pull is skipped so they are not overwritten."
            )
        else:
            print(f"Updating branch {BRANCH}...")
            subprocess.run(
                ["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", BRANCH],
                check=True,
            )
else:
    REPO_ROOT = Path.cwd().resolve()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / "pyproject.toml").is_file():
        raise FileNotFoundError(
            "Blackbox-Detection repository not found. "
            "Run this notebook inside the repository."
        )

os.chdir(REPO_ROOT)

# ============================================================
# IMPORTANT: Colab environment policy
# ============================================================
# Do NOT run `pip install -e .` with project dependencies in a live Colab
# kernel. pyproject.toml pins NumPy/SciPy/PyTorch versions; changing those
# binary packages after the kernel has already imported them can leave a
# mixed/broken scientific stack (e.g. `No module named numpy.rec`).
#
# Instead:
#   1) keep Colab's preinstalled NumPy/SciPy/PyTorch stack,
#   2) install only the extra packages Stage 1 needs,
#   3) install this repository editable with --no-deps.

COLAB_EXTRAS = [
    "av>=15,<17",
    "timm==1.0.15",
    "fvcore==0.1.5.post20221221",
    "iopath==0.1.10",
    "yacs==0.1.8",
    "einops==0.8.1",
    "transformers==4.57.6",
    "accelerate==1.9.0",
    "huggingface-hub==0.34.4",
    "safetensors==0.6.2",
    "sentencepiece==0.2.0",
    "tokenizers>=0.22,<0.24",
    "omegaconf==2.3.0",
    "hydra-core==1.3.2",
    "wandb==0.29.0",
    "easydict==1.13",
]

if IN_COLAB:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade-strategy",
            "only-if-needed",
            *COLAB_EXTRAS,
        ],
        check=True,
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "-e",
        str(REPO_ROOT),
    ],
    check=True,
)

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

# Import the scientific stack only AFTER package setup.
try:
    import numpy as np
    import pandas as pd
    import scipy
    from scipy.ndimage import maximum_filter
    import torch
    import yaml
except Exception as exc:
    raise RuntimeError(
        "The Colab scientific Python stack is inconsistent. "
        "This usually happens if NumPy/SciPy were changed earlier in the same "
        "runtime. Use Runtime -> Restart session once, then run this notebook "
        "again from the top. Do not run the old notebook's `pip install -e .` "
        "cell before restarting."
    ) from exc

from blackbox_detection.utils import (
    finish_wandb,
    init_wandb,
    load_checkpoint,
    seed_everything,
    setup_logger,
)

# ============================================================
# Persistent Google Drive paths
# ============================================================
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATASET_ROOT = DRIVE_PROJECT_ROOT / "DATASET"
DLC_ROOT = DATASET_ROOT / "DLC-2021"
CCD_ROOT = DATASET_ROOT / "CCD"

DLC_SPLIT_CSV = DLC_ROOT / "dlc_split.csv"
CCD_SPLIT_CSV = CCD_ROOT / "ccd_split.csv"

OUTPUT_ROOT = DRIVE_PROJECT_ROOT / "outputs" / "stage1"
WANDB_KEY_PATH = DRIVE_PROJECT_ROOT / "wandb_key.txt"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    required_paths = {
        "DATASET_ROOT": DATASET_ROOT,
        "DLC_ROOT": DLC_ROOT,
        "CCD_ROOT": CCD_ROOT,
        "DLC_SPLIT_CSV": DLC_SPLIT_CSV,
        "CCD_SPLIT_CSV": CCD_SPLIT_CSV,
        "WANDB_KEY_PATH": WANDB_KEY_PATH,
    }
    missing = [
        f"{name}: {path}"
        for name, path in required_paths.items()
        if not path.exists()
    ]
    if missing:
        raise FileNotFoundError(
            "Required Drive paths are missing:\n  " + "\n  ".join(missing)
        )

# ============================================================
# Weights & Biases login
# ============================================================
WANDB_ENABLED = True
WANDB_PROJECT = "blackbox-stage1"
WANDB_ENTITY = os.getenv("WANDB_ENTITY") or None
WANDB_MODE = os.getenv("WANDB_MODE") or None

if WANDB_ENABLED:
    import wandb

    if IN_COLAB:
        wandb_key = WANDB_KEY_PATH.read_text(encoding="utf-8").strip()
        if not wandb_key:
            raise ValueError(f"W&B key file is empty: {WANDB_KEY_PATH}")
        wandb.login(key=wandb_key, relogin=False)
        del wandb_key
    else:
        wandb.login()

GIT_COMMIT = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print("REPO_ROOT          :", REPO_ROOT)
print("branch / commit    :", BRANCH, "/", GIT_COMMIT)
print("DATASET_ROOT       :", DATASET_ROOT)
print("DLC split          :", DLC_SPLIT_CSV)
print("CCD split          :", CCD_SPLIT_CSV)
print("persistent outputs :", OUTPUT_ROOT)
print("numpy              :", np.__version__)
print("scipy              :", scipy.__version__)
print("torch              :", torch.__version__, "| cuda:", torch.cuda.is_available())
print("W&B                :", "enabled" if WANDB_ENABLED else "disabled")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Updating branch stage1-sangchun...


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sangchun1 (sangchun1-chung-ang-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


REPO_ROOT          : /content/Blackbox-Detection
branch / commit    : stage1-sangchun / df820fd
DATASET_ROOT       : /content/drive/MyDrive/Blackbox-Detection/DATASET
DLC split          : /content/drive/MyDrive/Blackbox-Detection/DATASET/DLC-2021/dlc_split.csv
CCD split          : /content/drive/MyDrive/Blackbox-Detection/DATASET/CCD/ccd_split.csv
persistent outputs : /content/drive/MyDrive/Blackbox-Detection/outputs/stage1
numpy              : 2.0.2
scipy              : 1.16.2
torch              : 2.8.0+cu126 | cuda: True
W&B                : enabled


In [2]:
from blackbox_detection.stage1.dataset import Stage1VideoDataset, build_dataloader, video_batch_adapter
from blackbox_detection.stage1.evaluator import AggregationConfig, Stage1Evaluator, save_predictions
from blackbox_detection.stage1.models import build_stage1_model, count_parameters
from blackbox_detection.stage1.sampling import build_clip_sampler
from blackbox_detection.stage1.trainer import Stage1Trainer, TrainConfig
from blackbox_detection.stage1.transforms import ClipAugmentConfig, build_video_transforms

logger = setup_logger("stage1.videomaev2_b")

## 2. Paths

In [3]:
CONFIG_DIR = REPO_ROOT / "configs" / "stage1"

# ============================================================
# Experiment data mode
# ============================================================
# A-series baseline:
TRAIN_DATA_MODE = "DLC"

# B-series hard-negative experiment:
# TRAIN_DATA_MODE = "DLC_CCD_OR"
#
# Current CCD has no physically re-recorded videos, so every CCD row is treated
# as Stage 1 ORIGINAL. To avoid letting 3,150 CCD train videos dominate the
# 483-video DLC train split, B-series defaults to a deterministic subset.
CCD_TRAIN_MAX = 200       # only used for DLC_CCD_OR; set None to use all CCD-train
CCD_SAMPLE_SEED = 42

if TRAIN_DATA_MODE not in {"DLC", "DLC_CCD_OR"}:
    raise ValueError("TRAIN_DATA_MODE must be 'DLC' or 'DLC_CCD_OR'.")

if TRAIN_DATA_MODE == "DLC":
    RUN_VARIANT = "dlc"
else:
    ccd_suffix = "all" if CCD_TRAIN_MAX is None else str(int(CCD_TRAIN_MAX))
    RUN_VARIANT = f"dlc_ccd_or_{ccd_suffix}"

print("train data mode :", TRAIN_DATA_MODE)
print("run variant     :", RUN_VARIANT)
print("DLC split       :", DLC_SPLIT_CSV)
print("CCD split       :", CCD_SPLIT_CSV)


train data mode : DLC
run variant     : dlc
DLC split       : /content/drive/MyDrive/Blackbox-Detection/DATASET/DLC-2021/dlc_split.csv
CCD split       : /content/drive/MyDrive/Blackbox-Detection/DATASET/CCD/ccd_split.csv


## 3. Config

In [4]:
CONFIG = yaml.safe_load((CONFIG_DIR / 'videomaev2_b.yaml').read_text(encoding='utf-8'))
MODEL_NAME = CONFIG['model']['name']
ADAPTER = video_batch_adapter()
SEED = int(CONFIG['train']['seed'])
RUN_DIR = OUTPUT_ROOT / RUN_VARIANT / MODEL_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
seed_everything(SEED, deterministic=False)
print(MODEL_NAME, '->', RUN_DIR)
print(json.dumps(CONFIG['model'], indent=2))

videomaev2_b -> /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc/videomaev2_b
{
  "name": "videomaev2_b",
  "finetune_mode": "last_n",
  "unfreeze_last_n": 4,
  "params": {
    "hf_model_id": "OpenGVLab/VideoMAEv2-Base",
    "backend": "hf_remote",
    "pretrained_path": null,
    "local_files_only": false,
    "num_frames": 16,
    "input_size": 224,
    "num_classes": 2,
    "dropout": 0.0
  }
}


## 4. Data from `dlc_split.csv` + `ccd_split.csv`

`dlc_split.csv` defines DLC train/val/test. `ccd_split.csv` supplies current CCD ORIGINAL hard negatives only when `TRAIN_DATA_MODE='DLC_CCD_OR'`.


In [5]:
VIDEO_EXTENSIONS = {
    ".mp4",
    ".mov",
    ".avi",
    ".mkv",
    ".m4v",
    ".webm",
}


def _normalize_rel_text(value: str) -> str:
    return str(value).replace("\\", "/").strip().lstrip("./")


def _build_dlc_video_index() -> dict[tuple[str, str], str]:
    """
    Index actual DLC video files under:
        DLC-2021/or/clips_video/**
        DLC-2021/re/clips_video/**

    `dlc_split.csv` contains clip_id (e.g. alb_id/00.or0001), but no video_path.
    We therefore resolve clip_id against the real files instead of hardcoding
    an extension or one exact nesting layout.
    """
    index: dict[tuple[str, str], str] = {}
    source_counts: dict[str, int] = {}

    for source in ("or", "re"):
        clips_root = DLC_ROOT / source / "clips_video"
        if not clips_root.is_dir():
            raise FileNotFoundError(f"DLC clips directory not found: {clips_root}")

        count = 0
        for path in clips_root.rglob("*"):
            if not path.is_file() or path.suffix.lower() not in VIDEO_EXTENSIONS:
                continue

            rel_no_suffix = path.relative_to(clips_root).with_suffix("").as_posix()
            key = (source, _normalize_rel_text(rel_no_suffix))

            if key in index and index[key] != str(path):
                raise ValueError(
                    "Duplicate DLC relative video key detected: "
                    f"{key} -> {index[key]} and {path}"
                )

            index[key] = str(path)
            count += 1

        source_counts[source] = count

    print("indexed DLC videos:", source_counts)
    if sum(source_counts.values()) == 0:
        raise FileNotFoundError(
            "No DLC video files were found below or/clips_video and re/clips_video. "
            "Check the mounted DLC-2021 directory."
        )

    return index


DLC_VIDEO_INDEX = _build_dlc_video_index()


def _resolve_dlc_video(source: str, clip_id: str) -> str:
    source = str(source).strip().lower()
    clip_id = _normalize_rel_text(clip_id)

    exact_key = (source, clip_id)
    if exact_key in DLC_VIDEO_INDEX:
        return DLC_VIDEO_INDEX[exact_key]

    # Robust fallback for an extra folder level around the clip.
    matches = []
    for (indexed_source, rel_key), video_path in DLC_VIDEO_INDEX.items():
        if indexed_source != source:
            continue
        if (
            rel_key.startswith(clip_id + "/")
            or rel_key.endswith("/" + clip_id)
            or rel_key == clip_id
        ):
            matches.append(video_path)

    if len(matches) == 1:
        return matches[0]

    # Last fallback: filename-like final token, but only when unique in source.
    leaf = Path(clip_id).name
    leaf_matches = [
        video_path
        for (indexed_source, rel_key), video_path in DLC_VIDEO_INDEX.items()
        if indexed_source == source and Path(rel_key).name == leaf
    ]
    if len(leaf_matches) == 1:
        return leaf_matches[0]

    raise FileNotFoundError(
        f"Could not uniquely resolve DLC clip_id={clip_id!r}, source={source!r}. "
        f"exact={exact_key in DLC_VIDEO_INDEX}, "
        f"prefix/suffix matches={len(matches)}, leaf matches={len(leaf_matches)}"
    )


def load_dlc_manifest(split_name: str) -> pd.DataFrame:
    raw = pd.read_csv(DLC_SPLIT_CSV).copy()

    required = {
        "clip_id",
        "class",
        "source",
        "document_type",
        "document_id",
        "group",
        "split",
        "device",
        "condition",
    }
    missing = sorted(required - set(raw.columns))
    if missing:
        raise ValueError(f"dlc_split.csv missing columns: {missing}")

    raw["split"] = raw["split"].astype(str).str.strip().str.lower()
    raw["source"] = raw["source"].astype(str).str.strip().str.lower()
    raw["class"] = raw["class"].astype(str).str.strip().str.lower()

    valid_sources = {"or", "re"}
    if not set(raw["source"]).issubset(valid_sources):
        raise ValueError(
            f"Unexpected DLC source values: {sorted(set(raw['source']) - valid_sources)}"
        )

    label_map = {
        "original": "ORIGINAL",
        "rerecorded": "RERECORDED",
    }
    if not set(raw["class"]).issubset(label_map):
        raise ValueError(
            f"Unexpected DLC class values: {sorted(set(raw['class']) - set(label_map))}"
        )

    frame = raw.loc[raw["split"].eq(split_name)].copy()
    if frame.empty:
        raise ValueError(f"No DLC rows for split={split_name!r}")

    frame["label"] = frame["class"].map(label_map)
    frame["video_id"] = "dlc__" + frame["clip_id"].astype(str).str.replace("/", "__", regex=False)
    frame["dataset"] = "dlc2021"
    frame["scene_type"] = "document"
    frame["video_path"] = [
        _resolve_dlc_video(source, clip_id)
        for source, clip_id in zip(frame["source"], frame["clip_id"])
    ]

    keep = [
        "video_path",
        "label",
        "video_id",
        "dataset",
        "scene_type",
        "clip_id",
        "source",
        "document_type",
        "document_id",
        "group",
        "device",
        "condition",
    ]
    return frame[keep].reset_index(drop=True)


def _relocate_ccd_path(value: str) -> str:
    """
    ccd_split.csv currently stores Colab absolute paths. If the Drive root moves,
    rebuild the path from the DATASET/ suffix instead of editing the CSV.
    """
    raw = str(value).strip()
    p = Path(raw)

    if p.is_file():
        return str(p)

    normalized = raw.replace("\\", "/")
    marker = "/DATASET/"
    if marker in normalized:
        relative = normalized.split(marker, 1)[1]
        candidate = DATASET_ROOT / relative
        return str(candidate)

    if normalized.startswith("DATASET/"):
        return str(DATASET_ROOT / normalized[len("DATASET/"):])

    if normalized.startswith("CCD/"):
        return str(DATASET_ROOT / normalized)

    return str(CCD_ROOT / normalized)


def load_ccd_manifest(split_name: str) -> pd.DataFrame:
    raw = pd.read_csv(CCD_SPLIT_CSV).copy()

    required = {"video_id", "video_path", "class", "split"}
    missing = sorted(required - set(raw.columns))
    if missing:
        raise ValueError(f"ccd_split.csv missing columns: {missing}")

    raw["split"] = raw["split"].astype(str).str.strip().str.lower()
    raw["class"] = raw["class"].astype(str).str.strip().str.lower()

    valid_scene_classes = {"crash", "normal"}
    if not set(raw["class"]).issubset(valid_scene_classes):
        raise ValueError(
            f"Unexpected CCD class values: "
            f"{sorted(set(raw['class']) - valid_scene_classes)}"
        )

    frame = raw.loc[raw["split"].eq(split_name)].copy()
    if frame.empty:
        raise ValueError(f"No CCD rows for split={split_name!r}")

    # IMPORTANT: Crash/Normal are scene categories, not Stage 1 labels.
    frame["scene_type"] = frame["class"]
    frame["label"] = "ORIGINAL"
    frame["dataset"] = "ccd"
    frame["video_id"] = "ccd__" + frame["video_id"].astype(str).str.strip()
    frame["video_path"] = frame["video_path"].map(_relocate_ccd_path)

    keep = [
        "video_path",
        "label",
        "video_id",
        "dataset",
        "scene_type",
    ]
    return frame[keep].reset_index(drop=True)


def sample_ccd_hard_negatives(
    frame: pd.DataFrame,
    max_videos: int | None,
    seed: int,
) -> pd.DataFrame:
    if max_videos is None or max_videos >= len(frame):
        return frame.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    if max_videos <= 0:
        return frame.iloc[0:0].copy()

    # Preserve Crash:Normal proportions as closely as possible.
    counts = frame["scene_type"].value_counts().sort_index()
    raw_targets = counts / counts.sum() * int(max_videos)
    targets = np.floor(raw_targets).astype(int)

    remaining = int(max_videos) - int(targets.sum())
    if remaining > 0:
        fractional = (raw_targets - targets).sort_values(ascending=False)
        for scene_type in fractional.index[:remaining]:
            targets.loc[scene_type] += 1

    sampled = []
    for scene_type, n in targets.items():
        group = frame.loc[frame["scene_type"].eq(scene_type)]
        sampled.append(group.sample(n=int(n), random_state=seed))

    return (
        pd.concat(sampled, ignore_index=True)
        .sample(frac=1.0, random_state=seed)
        .reset_index(drop=True)
    )


# ============================================================
# Build the actual Stage 1 manifests from the two supplied CSVs.
# ============================================================
dlc_train_df = load_dlc_manifest("train")
dlc_val_df = load_dlc_manifest("val")

ccd_train_full_df = load_ccd_manifest("train")

if TRAIN_DATA_MODE == "DLC":
    ccd_train_df = ccd_train_full_df.iloc[0:0].copy()
    train_df = dlc_train_df.copy()
else:
    ccd_train_df = sample_ccd_hard_negatives(
        ccd_train_full_df,
        CCD_TRAIN_MAX,
        CCD_SAMPLE_SEED,
    )
    train_df = pd.concat(
        [dlc_train_df, ccd_train_df],
        ignore_index=True,
    ).sample(frac=1.0, random_state=CCD_SAMPLE_SEED).reset_index(drop=True)

# Current primary validation is always DLC OR/RE.
val_df = dlc_val_df.copy()

# Basic split sanity.
video_overlap = set(train_df["video_id"]) & set(val_df["video_id"])
if video_overlap:
    raise ValueError(f"train/val video_id leakage: {sorted(video_overlap)[:5]}")

if train_df["label"].nunique() < 2:
    raise ValueError("Training data must contain both ORIGINAL and RERECORDED.")
if val_df["label"].nunique() < 2:
    raise ValueError("DLC validation must contain both ORIGINAL and RERECORDED.")

# Check only paths actually used in this run.
missing_train = [p for p in train_df["video_path"] if not Path(p).is_file()]
missing_val = [p for p in val_df["video_path"] if not Path(p).is_file()]
if missing_train or missing_val:
    raise FileNotFoundError(
        f"Missing video files: train={len(missing_train)}, val={len(missing_val)}. "
        f"Examples: {(missing_train + missing_val)[:5]}"
    )

ccd_share = (
    float((train_df["dataset"] == "ccd").mean())
    if len(train_df)
    else 0.0
)

print("\nDLC split counts")
dlc_split_preview = pd.read_csv(DLC_SPLIT_CSV)
display(pd.crosstab(dlc_split_preview["split"], dlc_split_preview["class"], margins=True))

print("\nCCD split counts")
ccd_split_preview = pd.read_csv(CCD_SPLIT_CSV)
display(pd.crosstab(ccd_split_preview["split"], ccd_split_preview["class"], margins=True))

print("\nACTUAL TRAIN dataset x Stage 1 label")
display(pd.crosstab(train_df["dataset"], train_df["label"], margins=True))

if len(ccd_train_df):
    print("\nCCD hard-negative scene composition")
    display(ccd_train_df["scene_type"].value_counts().rename("count").to_frame())

print("\nPRIMARY VAL")
display(pd.crosstab(val_df["dataset"], val_df["label"], margins=True))

print(
    f"train={len(train_df)} | DLC train={len(dlc_train_df)} | "
    f"CCD train used={len(ccd_train_df)} | CCD share={ccd_share:.1%} | "
    f"DLC val={len(val_df)}"
)

# dlc-2021_or.csv and dlc-2021_re.csv are deliberately NOT used here.
# They are metadata/list files and are not the fixed Stage 1 train/val/test split.


indexed DLC videos: {'or': 290, 're': 400}

DLC split counts


class,original,rerecorded,All
split,,,
test,43,60,103
train,203,280,483
val,44,60,104
All,290,400,690



CCD split counts


class,crash,normal,All
split,,,
test,225,450,675
train,1050,2100,3150
val,225,450,675
All,1500,3000,4500



ACTUAL TRAIN dataset x Stage 1 label


label,ORIGINAL,RERECORDED,All
dataset,,,
dlc2021,203,280,483
All,203,280,483



PRIMARY VAL


label,ORIGINAL,RERECORDED,All
dataset,,,
dlc2021,44,60,104
All,44,60,104


train=483 | DLC train=483 | CCD train used=0 | CCD share=0.0% | DLC val=104


In [6]:
# ============================================================
# Start one W&B run for this experiment
# ============================================================
TRAIN_DATA_TAG = RUN_VARIANT.upper()
RUN_NAME = f"{MODEL_NAME}__{RUN_VARIANT}__seed{SEED}__{GIT_COMMIT}"

# Safe for notebook re-runs.
finish_wandb()

wandb_run = init_wandb(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=RUN_NAME,
    group=RUN_VARIANT,
    tags=["stage1", "video", MODEL_NAME, RUN_VARIANT],
    config={
        "model_name": MODEL_NAME,
        "branch": "video",
        "train_data_mode": TRAIN_DATA_MODE,
        "run_variant": RUN_VARIANT,
        "seed": SEED,
        "git_commit": GIT_COMMIT,
        "dlc_split_csv": str(DLC_SPLIT_CSV),
        "ccd_split_csv": str(CCD_SPLIT_CSV),
        "num_dlc_train_videos": int(len(dlc_train_df)),
        "num_ccd_train_videos": int(len(ccd_train_df)),
        "num_train_videos": int(len(train_df)),
        "num_primary_val_videos": int(len(val_df)),
        "ccd_train_share": float(ccd_share),
        "ccd_train_max": None if CCD_TRAIN_MAX is None else int(CCD_TRAIN_MAX),
        "model_config": CONFIG.get("model", {}),
        "train_config": CONFIG.get("train", {}),
        "data_config": CONFIG.get("data", {}),
    },
    directory=RUN_DIR / "wandb",
    mode=WANDB_MODE,
)

print("W&B run:", wandb_run.name)
print("W&B url :", wandb_run.url)


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


W&B run: videomaev2_b__dlc__seed42__df820fd
W&B url : https://wandb.ai/sangchun1-chung-ang-university/blackbox-stage1/runs/yctdezrs


## 5. Model

In [7]:
model = build_stage1_model(
    MODEL_NAME,
    finetune_mode=CONFIG['model']['finetune_mode'],
    unfreeze_last_n=int(CONFIG['model']['unfreeze_last_n']),
    **CONFIG['model']['params'],
)
print('blocks:', len(model.blocks), '| feature dim:', model.feature_dim)
print('parameters:', count_parameters(model))
print('load report:', getattr(model, 'load_report', None))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


blocks: 12 | feature dim: 768
parameters: {'total': 86228738, 'trainable': 28349954, 'frozen': 57878784}
load report: {'source': 'OpenGVLab/VideoMAEv2-Base', 'backend': 'hf_remote', 'weights': {'source': 'OpenGVLab/VideoMAEv2-Base'}}


### 5.1 Datasets and loaders

In [8]:
video_config = CONFIG['data']
augmentation_config = CONFIG['augmentation']
preprocessing = model.preprocessing()
print('checkpoint preprocessing:', preprocessing)

train_transform, val_transform = build_video_transforms(
    crop_size=int(preprocessing['input_size']),
    mean=tuple(preprocessing['mean']),
    std=tuple(preprocessing['std']),
    train_config=ClipAugmentConfig(
        crop_size=int(preprocessing['input_size']),
        scale_range=tuple(augmentation_config['scale_range']),
        ratio_range=tuple(augmentation_config['ratio_range']),
        hflip_prob=float(augmentation_config['hflip_prob']),
        brightness=float(augmentation_config['brightness']),
        contrast=float(augmentation_config['contrast']),
        perspective_prob=float(augmentation_config['perspective_prob']),
        perspective_scale=float(augmentation_config['perspective_scale']),
    ),
)

train_dataset = Stage1VideoDataset(
    train_df,
    clip_sampler=build_clip_sampler(train=True, num_frames=int(video_config['num_frames']), strides=video_config['train_strides'], num_clips=int(video_config['train_num_clips'])),
    transform=train_transform, on_error='zero', deterministic=False,
)
val_dataset = Stage1VideoDataset(
    val_df,
    clip_sampler=build_clip_sampler(train=False, num_frames=int(video_config['num_frames']), val_stride=int(video_config['val_stride']), num_clips=int(video_config['val_num_clips'])),
    transform=val_transform, on_error='zero', deterministic=True,
)
train_loader = build_dataloader(train_dataset, batch_size=int(video_config['batch_size']), shuffle=True, num_workers=int(video_config['num_workers']), seed=SEED, drop_last=True)
val_loader = build_dataloader(val_dataset, batch_size=int(video_config['val_batch_size']), shuffle=False, num_workers=int(video_config['num_workers']), seed=SEED)

batch = next(iter(train_loader))
print('clip batch:', tuple(batch['pixels'].shape), '| labels:', batch['label'].tolist())

checkpoint preprocessing: {'input_kind': 'video', 'num_frames': 16, 'input_size': 224, 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'channels_first': True, 'source': 'OpenGVLab/VideoMAEv2-Base', 'notes': 'Official checkpoint preprocessing: ImageNet normalisation, 16 frames, 224x224, tubelet_size=2.'}
clip batch: (4, 1, 3, 16, 224, 224) | labels: [0, 1, 1, 1]


## 6. Training

In [9]:
train_config = CONFIG['train']
trainer_config = TrainConfig(
    epochs=int(train_config['epochs']),
    learning_rate=float(train_config['learning_rate']),
    head_learning_rate=(
        float(train_config['head_learning_rate'])
        if train_config.get('head_learning_rate') is not None
        else None
    ),
    weight_decay=float(train_config['weight_decay']),
    warmup_ratio=float(train_config['warmup_ratio']),
    grad_accum_steps=int(train_config['grad_accum_steps']),
    max_grad_norm=float(train_config['max_grad_norm']),
    amp=bool(train_config['amp']),
    label_smoothing=float(train_config.get('label_smoothing', 0.0)),
    early_stopping_patience=int(train_config['early_stopping_patience']),
    eval_every=int(train_config['eval_every']),
    seed=SEED,
    output_dir=RUN_DIR,
    model_name=MODEL_NAME,
    wandb_enabled=WANDB_ENABLED,
)

trainer = Stage1Trainer(
    model,
    trainer_config,
    adapter=ADAPTER,
    aggregation=AggregationConfig(
        frame_method=CONFIG['evaluation']['aggregation']['frame_method'],
        video_method=CONFIG['evaluation']['aggregation']['video_method'],
    ),
    model_config={'name': MODEL_NAME, 'params': CONFIG['model']['params']},
)

print('device:', trainer.device, '| amp:', trainer.amp)
outcome = trainer.fit(train_loader, val_loader)
print(
    f'best epoch {outcome.best_epoch}: Macro-F1 {outcome.best_macro_f1:.4f} '
    f'at threshold {outcome.best_threshold:.3f}'
)

device: cuda | amp: True


Train 1/10:   0%|          | 0/120 [00:00<?, ?it/s]

[2026-09-14 10:59:22] INFO | blackbox_detection.stage1.videomaev2_b | Epoch 1 | loss 0.5636 | val Macro-F1 0.8453 @ thr 0.439 (0.5: 0.8203)  <- best


Train 2/10:   0%|          | 0/120 [00:00<?, ?it/s]

[2026-09-14 11:10:22] INFO | blackbox_detection.stage1.videomaev2_b | Epoch 2 | loss 0.1600 | val Macro-F1 0.8930 @ thr 0.765 (0.5: 0.8716)  <- best


...

[2026-09-14 12:17:51] INFO | blackbox_detection.stage1.videomaev2_b | Epoch 8 | loss 0.0139 | val Macro-F1 0.9610 @ thr 0.996 (0.5: 0.9503)


Train 9/10:   0%|          | 0/120 [00:00<?, ?it/s]

[2026-09-14 12:28:39] INFO | blackbox_detection.stage1.videomaev2_b | Epoch 9 | loss 0.0035 | val Macro-F1 0.9608 @ thr 0.958 (0.5: 0.9503)


Train 10/10:   0%|          | 0/120 [00:00<?, ?it/s]

[2026-09-14 12:39:32] INFO | blackbox_detection.stage1.videomaev2_b | Epoch 10 | loss 0.0009 | val Macro-F1 0.9608 @ thr 0.957 (0.5: 0.9503)
[2026-09-14 12:39:34] INFO | blackbox_detection.stage1.videomaev2_b | Early stopping after 4 epoch(s) without improvement.
[2026-09-14 12:39:34] INFO | blackbox_detection.stage1.videomaev2_b | Best epoch 6 with validation Macro-F1 0.9704 at threshold 0.961.
best epoch 6: Macro-F1 0.9704 at threshold 0.961


## 7. Validation & save

In [10]:
load_checkpoint(
    RUN_DIR / 'best.pt',
    model=model,
    map_location=trainer.device,
    restore_rng_state=False,
)

evaluator = Stage1Evaluator(
    model,
    ADAPTER,
    device=trainer.device,
    amp=trainer.amp,
    aggregation=trainer.aggregation,
)
result, units = evaluator.evaluate(val_loader, return_units=True)

print(f'Macro-F1            : {result.macro_f1:.4f}')
print(f'Macro-F1 @ thr 0.5  : {result.macro_f1_at_default:.4f}')
print(f'optimal threshold   : {result.threshold:.4f}')
print(f'class-wise F1       : {result.per_class_f1}')
print(f'per-dataset Macro-F1: {result.dataset_scores}')
print(f'videos              : {result.num_videos} ({result.num_invalid_videos} with decode problems)')

save_predictions(result.predictions, RUN_DIR / 'val_predictions.csv')
outcome.history.to_csv(RUN_DIR / 'history.csv', index=False)
summary = {
    'model_name': MODEL_NAME,
    'val_macro_f1': float(result.macro_f1),
    'val_macro_f1_at_0.5': float(result.macro_f1_at_default),
    'best_threshold': float(result.threshold),
    'per_class_f1': result.per_class_f1,
    'best_epoch': int(outcome.best_epoch),
    'num_val_videos': int(result.num_videos),
    'preprocessing': dict(model.preprocessing()),
}
(RUN_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')
print('saved to:', RUN_DIR)


# Record final/best values in W&B summary as the experiment table source of truth.
if WANDB_ENABLED and wandb_run is not None:
    wandb_run.summary["best_epoch"] = int(outcome.best_epoch)
    wandb_run.summary["best_val_macro_f1"] = float(result.macro_f1)
    wandb_run.summary["best_val_macro_f1_at_0.5"] = float(result.macro_f1_at_default)
    wandb_run.summary["best_threshold"] = float(result.threshold)
    wandb_run.summary["num_val_videos"] = int(result.num_videos)
    wandb_run.summary["git_commit"] = GIT_COMMIT
    for class_name, score in result.per_class_f1.items():
        wandb_run.summary[f"best_f1_{class_name.lower()}"] = float(score)

    # Save lightweight result files to the W&B run as well.
    try:
        wandb_run.save(str(RUN_DIR / "summary.json"), base_path=str(RUN_DIR))
        wandb_run.save(str(RUN_DIR / "history.csv"), base_path=str(RUN_DIR))
        wandb_run.save(str(RUN_DIR / "val_predictions.csv"), base_path=str(RUN_DIR))
    except Exception as exc:
        print("W&B file upload warning:", exc)

finish_wandb()
print("W&B run finished.")


Macro-F1            : 0.9704
Macro-F1 @ thr 0.5  : 0.9401
optimal threshold   : 0.9608
class-wise F1       : {'ORIGINAL': 0.9655172413793104, 'RERECORDED': 0.9752066115702479}
per-dataset Macro-F1: {'dlc2021': 0.9703619264747791}
videos              : 104 (0 with decode problems)


wandb: WARNING Copied 1 file (cross-volume or links unavailable). Downgrading policy to 'now' for those files because live updates won't propagate from the originals. Re-run wandb.save to resync, or place your run directory on the same drive to enable hardlinks.
wandb: WARNING Copied 1 file (cross-volume or links unavailable). Downgrading policy to 'now' for those files because live updates won't propagate from the originals. Re-run wandb.save to resync, or place your run directory on the same drive to enable hardlinks.
wandb: WARNING Copied 1 file (cross-volume or links unavailable). Downgrading policy to 'now' for those files because live updates won't propagate from the originals. Re-run wandb.save to resync, or place your run directory on the same drive to enable hardlinks.


saved to: /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc/videomaev2_b


train/loss,█▃▂▂▂▁▁▁▁▁
val/f1_ORIGINAL,▁▄▇▇▇█████
val/f1_RERECORDED,▁▂▇▇▇█▇▇▇▇
val/macro_f1,▁▄▇▇▇█▇▇▇▇
val/macro_f1@0.5,▁▄▅▇█▇████
val/macro_f1_dlc2021,▁▄▇▇▇█▇▇▇▇
val/num_invalid_videos,▁▁▁▁▁▁▁▁▁▁
val/threshold,▃▆█▁▃█████
best_epoch,6
best_f1_original,0.96552
best_f1_rerecorded,0.97521


W&B run finished.
